# Anthropic Tool Use with Claude

This notebook demonstrates how Claude handles tool use. Unlike OpenAI's function calling, Anthropic's approach focuses on the model explicitly requesting a tool and the system providing the output in a structured message exchange.

In [1]:
import os
import json
import pandas as pd
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Load real data for grounded examples
customers_df = pd.read_csv('customers.csv')
orders_df = pd.read_csv('order_mock_data.csv')

print("Data loaded successfully.")

Data loaded successfully.


## 1. Defining the Tools

Anthropic requires tools to be defined in a specific JSON schema format. We will use the same logic as the OpenAI example to maintain consistency in our data access.

In [2]:
def get_customer_info(customer_id: str = None, name: str = None):
    """Get customer details by customer_id OR by name.
    Orders only return the customer's name, so this tool accepts either."""
    if customer_id:
        match = customers_df[customers_df['customer_id'] == customer_id]
    elif name:
        match = customers_df[customers_df['name'].str.lower() == name.lower()]
    else:
        return {"error": "Provide customer_id or name"}
    if not match.empty:
        return match.to_dict('records')[0]
    return {"error": "Customer not found"}

In [3]:
def get_order_status(order_id):
    """Get the current status of an order. IDs look like '#ORDNO12'."""
    if not order_id.startswith("#"):
        order_id = "#" + order_id
    order = orders_df[orders_df['id'] == order_id]
    if not order.empty:
        return order.to_dict('records')[0]
    return {"error": "Order not found"}

In [4]:
tools = [
    {
        "name": "get_customer_info",
        "description": "Get customer details (tier, email, total spent). Accepts customer_id ('CUST005') OR full name ('Mark Perez'). Use 'name' when chaining from get_order_status, since orders only return the customer's name.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Customer ID like 'CUST005' (optional)."},
                "name": {"type": "string", "description": "Full customer name like 'Mark Perez' (optional)."}
            }
        }
    },
    {
        "name": "get_order_status",
        "description": "Get status, customer name, invoice, and items for an order. IDs look like '#ORDNO12'. Returns the customer's NAME (not customer_id) — pass that name to get_customer_info for tier/email.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID in the form '#ORDNO12' (leading '#' optional)."}
            },
            "required": ["order_id"]
        }
    }
]

## 2. The Tool Use Loop

Claude's loop is slightly different: it returns a `tool_use` content block. We must respond with a `tool_result` block.

In [5]:
MODEL = "claude-haiku-4-5-20251001"

available_functions = {
    "get_customer_info": get_customer_info,
    "get_order_status": get_order_status,
}

In [6]:
def run_conversation(user_prompt, max_iters=6):
    """Multi-turn loop: keep calling Claude until stop_reason != 'tool_use'."""
    messages = [{"role": "user", "content": user_prompt}]

    for _ in range(max_iters):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason != "tool_use":
            # Return the final text block
            return next((b.text for b in response.content if b.type == "text"), "")

        # Record Claude's turn (which includes tool_use blocks)
        messages.append({"role": "assistant", "content": response.content})

        # Execute every tool Claude asked for and send results back in one user turn
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"-> Calling {block.name}({block.input})")
                result = available_functions[block.name](**block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                })
        messages.append({"role": "user", "content": tool_results})

    return "[stopped: hit max_iters]"

In [7]:
# Test with a real order from order_mock_data.csv
query = "Who is the customer for order #ORDNO5 and what's the status?"
print(f"Query: {query}\n")
print(f"Response: {run_conversation(query)}")

Query: Who is the customer for order #ORDNO5 and what's the status?

-> Calling get_order_status({'order_id': '#ORDNO5'})
Response: The customer for order **#ORDNO5** is **Charles Martinez**.

**Order Status: Shipping**

Additional details:
- **Invoice Number:** INV-169622
- **Items:** Fitness Tracker, Desk Lamp


## 3. Building the Same Agent with LangChain `create_agent`

The raw Anthropic loop above shows the mechanics. In production, `create_agent` from `langchain.agents` gives you the same behaviour (tool binding + multi-turn loop + message history) in one call. Same `.env` key, same two tools, swapped to `ChatAnthropic`.

In [8]:
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent

@tool
def lc_get_customer_info(customer_id: str = "", name: str = "") -> dict:
    """Get customer details (tier, email, total spent).
    Accepts customer_id ('CUST005') OR full name ('Mark Perez').
    When chaining from lc_get_order_status, pass the returned name here."""
    return get_customer_info(
        customer_id=customer_id or None,
        name=name or None,
    )

@tool
def lc_get_order_status(order_id: str) -> dict:
    """Get status, customer name, invoice, and items for an order.
    IDs look like '#ORDNO12'. Returns the customer's NAME (not customer_id)."""
    return get_order_status(order_id)

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
llm = ChatAnthropic(model_name="claude-haiku-4-5-20251001")
agent = create_agent(llm, tools=[lc_get_customer_info, lc_get_order_status])

result = agent.invoke({
    "messages": [("user", "Who placed order #ORDNO30 and what tier are they?")]
})
print(result["messages"][-1].content)

**Michael Miller** placed order #ORDNO30, and he is a **Silver** tier customer.
